# 04 — Knowledge Distillation: AfroXLMR-Comet (Paper 3)

Reproduce the AfroXLMR-Comet distillation pipeline from arXiv:2502.18020.

**Teacher**: Davlan/afro-xlmr-large — 559.9M parameters, 2.09 GB

**Student**: AfroXLMR-Comet — 6 layers, 256 hidden, 8 heads — 68.9M params, 263 MB

**Loss**: L_total = 0.5 × KL_distill + 0.5 × MSE_attention

**Languages**: Kinyarwanda, Swahili, Hausa, Igbo, Yoruba

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import pandas as pd
import matplotlib.pyplot as plt

from mrm.models.distillation import AfroXLMRComet, TEACHER_MODEL, STUDENT_CONFIG
from mrm.models.transformer_clf import load_transformer_classifier
from mrm.training.distill_trainer import DistillationTrainer
from mrm.data.datasets import HFTextDataset
from mrm.evaluation.metrics import parameter_count, size_reduction_pct, speedup_ratio, benchmark_inference_speed
from transformers import AutoTokenizer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}')

## 1. Teacher vs Student Architecture Comparison

In [ ]:
teacher_stats = {'hidden_size': 1024, 'layers': 24, 'heads': 16, 'params_M': 559.9, 'size_MB': 2135.86}
student_stats = {**STUDENT_CONFIG, 'params_M': 68.9, 'size_MB': 262.99}

print('Architecture Comparison:')
print(f"{'Metric':<25} {'Teacher':>12} {'Student':>12} {'Reduction':>12}")
print('-' * 63)
for key in ['hidden_size', 'num_hidden_layers', 'num_attention_heads', 'intermediate_size']:
    t_val = teacher_stats.get(key, student_stats.get(key, 'N/A'))
    s_val = student_stats.get(key, 'N/A')
    print(f"{key:<25} {str(t_val):>12} {str(s_val):>12}")
print(f"{'Parameters (M)':<25} {teacher_stats['params_M']:>12.1f} {student_stats['params_M']:>12.1f} {size_reduction_pct(559_890_432, 68_937_216):>11.1f}%")
print(f"{'Model size (MB)':<25} {teacher_stats['size_MB']:>12.2f} {student_stats['size_MB']:>12.2f} {size_reduction_pct(2135860000, 262990000):>11.1f}%")
print(f"{'Inference (ms)':<25} {'293.9':>12} {'14.0':>12} {speedup_ratio(293.9, 14.0):>11.1f}x")

## 2. Initialize Teacher and Student

In [ ]:
print('Loading teacher model (this may take a few minutes)...')
teacher_model, _ = load_transformer_classifier(TEACHER_MODEL, num_labels=3)
teacher_model.eval()
print(f'Teacher parameters: {parameter_count(teacher_model):,}')

student = AfroXLMRComet.from_scratch(teacher_name=TEACHER_MODEL, num_labels=3)
print(f'Student parameters: {student.num_parameters:,}')
print(f'Size reduction: {size_reduction_pct(parameter_count(teacher_model), student.num_parameters):.2f}%')

## 3. Distillation Training

In [ ]:
LANGUAGE = 'kin'
tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL)

train_ds = HFTextDataset.from_csv(f'../data/processed/afrisenti/{LANGUAGE}_train.csv', tokenizer)
test_ds  = HFTextDataset.from_csv(f'../data/processed/afrisenti/{LANGUAGE}_test.csv', tokenizer)

train_dl = torch.utils.data.DataLoader(train_ds, batch_size=8, shuffle=True)
eval_dl  = torch.utils.data.DataLoader(test_ds,  batch_size=16)

trainer = DistillationTrainer(
    teacher_model=teacher_model,
    student_model=student,
    train_dataloader=train_dl,
    eval_dataloader=eval_dl,
    temperature=2.0,
    alpha=0.5,
    lr=5e-5,
    num_epochs=15,
    grad_accum_steps=2,
    patience=3,
    fp16=DEVICE.startswith('cuda'),
    device=DEVICE,
)
history = trainer.train()

## 4. Results: F1 Retention Across Languages

In [ ]:
# Paper 3 Table IV results
results = pd.DataFrame({
    'Language':    ['Kinyarwanda', 'Swahili', 'Hausa', 'Igbo', 'Yoruba', 'Average'],
    'Teacher F1':  [68.91,         64.10,     79.30,   79.60,  74.20,   73.22],
    'Base F1':     [62.54,         60.81,     78.27,   78.63,  70.25,   70.10],
    'Mini F1':     [60.23,         61.88,     74.29,   75.20,  63.10,   66.94],
    'Student F1':  [58.94,         55.30,     71.89,   73.82,  66.39,   65.28],
    'Retention%':  [85.5,          86.3,      90.6,    92.8,   89.5,    89.2],
})
print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(results) - 1)
for col, marker in [('Teacher F1', 'o'), ('Student F1', 's')]:
    ax.plot(x, results[col].iloc[:-1], marker=marker, label=col, linewidth=2)
ax.set_xticks(list(x))
ax.set_xticklabels(results['Language'].iloc[:-1])
ax.set_ylabel('Weighted F1 (%)')
ax.set_title('Teacher vs AfroXLMR-Comet Student — F1 by Language')
ax.legend()
ax.set_ylim(50, 85)
plt.tight_layout()
plt.savefig('../paper/figures/distillation_f1.pdf', bbox_inches='tight')
plt.show()